In [2]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN
from sklearn.metrics import classification_report
import joblib
import os

# Load the dataset
df = pd.read_csv("multiple.csv.csv")
df = df.dropna()  # Drop missing values if any

# Separate Features & Target
X = df.drop(columns=["Class"])  # All features used for prediction
y = df["Class"]                 # Actual labels (0 = normal, 1 = fraud)

# Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Save the Scaler
os.makedirs("models", exist_ok=True)
joblib.dump(scaler, "models/scaler.pkl")

# Train Isolation Forest
iso_forest = IsolationForest(n_estimators=100, contamination=0.01, random_state=42)
iso_forest.fit(X_scaled)

# Predict (1 = normal, -1 = fraud)
iso_preds = iso_forest.predict(X_scaled)
df['IsoForest_Prediction'] = [1 if p == -1 else 0 for p in iso_preds]  # Convert to 1 = fraud

# Train DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_scaled)

# Mark DBSCAN noise (-1) as fraud
df['DBSCAN_Cluster'] = dbscan_labels
df['DBSCAN_Fraud'] = (df['DBSCAN_Cluster'] == -1).astype(int)

# Evaluate Models
print("Isolation Forest Results:")
print(classification_report(y, df['IsoForest_Prediction']))

print("\n DBSCAN Results:")
print(classification_report(y, df['DBSCAN_Fraud']))

# Save Models
joblib.dump(iso_forest, "models/isolation_forest_model.pkl")
joblib.dump(dbscan, "models/dbscan_model.pkl")

print("\n Models and Scaler saved in 'models/' folder.")


Isolation Forest Results:
              precision    recall  f1-score   support

         0.0       1.00      0.99      1.00       119
         1.0       0.50      1.00      0.67         1

    accuracy                           0.99       120
   macro avg       0.75      1.00      0.83       120
weighted avg       1.00      0.99      0.99       120


 DBSCAN Results:
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00       119
         1.0       0.01      1.00      0.02         1

    accuracy                           0.01       120
   macro avg       0.00      0.50      0.01       120
weighted avg       0.00      0.01      0.00       120


 Models and Scaler saved in 'models/' folder.


c:\Users\OM\Desktop\CreditCard_Project\Credit-card-fraud-detection\venv\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\OM\Desktop\CreditCard_Project\Credit-card-fraud-detection\venv\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\OM\Desktop\CreditCard_Project\Credit-card-fraud-detection\venv\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` paramete